# 03 - Mining & Clustering
## Đề tài 11: Phân tích đánh giá khách sạn

**Mục tiêu:**
- Association Rules Mining
- Topic Clustering (K-Means)
- Silhouette Score evaluation

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_data
from src.data.cleaner import DataCleaner
from src.mining.association import AssociationMiner
from src.mining.clustering import ClusterAnalyzer
from src.visualization.plots import PlotGenerator

[WARNING] HDBSCAN not available. Install with: pip install hdbscan


## 1. Load and Prepare Data

In [2]:
df = load_data(n_rows=2000)
cleaner = DataCleaner()
df_cleaned, _ = cleaner.clean(df)
print(f"Cleaned data: {df_cleaned.shape}")

[WARNING] Config file not found at configs\params.yaml
Using default configuration...
[INFO] File not found at data\raw\hotel_reviews.csv
[INFO] Generating sample hotel reviews data...
[INFO] Generating 2000 sample hotel reviews...
[INFO] Generated 2000 sample reviews
[INFO] Rating distribution:
rating
1    147
2    208
3    579
4    477
5    589
Name: count, dtype: int64
DATA CLEANING PROCESS

[Step 1/5] Basic data cleaning...
  - Original rows: 2000

[Step 2/5] Handling missing values...

[Step 3/5] Text preprocessing...
[INFO] Cleaning text...

[Step 4/5] Validating ratings...

[Step 5/5] Adding derived features...

CLEANING SUMMARY
Original rows: 2000
Final rows: 2000
Rows removed: 0 (0.0%)
Columns: ['review_text', 'rating', 'sentiment', 'reviewer_name', 'date', 'hotel_name', 'original_text', 'original_length', 'original_word_count', 'cleaned_text', 'cleaned_length', 'cleaned_word_count', 'length_reduction', 'word_reduction', 'review_length', 'word_count', 'avg_word_length', 'sente

## 2. Association Rules Mining

In [3]:
# Mine association rules
miner = AssociationMiner()
rules = miner.mine_association_rules(df_cleaned, text_column='cleaned_text')

if len(rules) > 0:
    print(f"\nFound {len(rules)} rules")
    miner.visualize_top_rules(10)
    
    # Interpretations
    print("\n--- RULE INTERPRETATIONS ---")
    for interp in miner.interpret_rules()[:5]:
        print(f"\n{interp['rule']}")
        print(f"  Meaning: {interp['meaning']}")

[INFO] Mining association rules...
[INFO] Generated 1846 transactions
[INFO] Finding frequent itemsets with min_support=0.01...
[INFO] Found 51 frequent itemsets
[INFO] Generated 108 association rules

Found 108 rules

TOP ASSOCIATION RULES

Rule 36:
  IF ASPECT_PRICE
  THEN ASPECT_SERVICE, ASPECT_AMENITIES
  Support: 0.0320
  Confidence: 0.2235
  Lift: 6.07

Rule 35:
  IF ASPECT_SERVICE, ASPECT_AMENITIES
  THEN ASPECT_PRICE
  Support: 0.0320
  Confidence: 0.8676
  Lift: 6.07

Rule 59:
  IF ASPECT_SERVICE, ASPECT_CLEANLINESS
  THEN ASPECT_FOOD
  Support: 0.0406
  Confidence: 0.4870
  Lift: 4.50

Rule 60:
  IF ASPECT_FOOD
  THEN ASPECT_SERVICE, ASPECT_CLEANLINESS
  Support: 0.0406
  Confidence: 0.3750
  Lift: 4.50

Rule 61:
  IF ASPECT_SERVICE
  THEN ASPECT_FOOD, ASPECT_CLEANLINESS
  Support: 0.0406
  Confidence: 0.1168
  Lift: 2.88

Rule 58:
  IF ASPECT_FOOD, ASPECT_CLEANLINESS
  THEN ASPECT_SERVICE
  Support: 0.0406
  Confidence: 1.0000
  Lift: 2.88

Rule 52:
  IF ASPECT_FOOD, ASPECT_

## 3. Topic Clustering

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF features
vectorizer = TfidfVectorizer(max_features=1000, max_df=0.95, min_df=2)
X = vectorizer.fit_transform(df_cleaned['cleaned_text'].astype(str))

# Cluster
cluster_analyzer = ClusterAnalyzer()
labels, coords_2d = cluster_analyzer.fit_predict(X.toarray(), method='kmeans')

print(f"\nSilhouette Score: {cluster_analyzer.silhouette_avg:.4f}")

[INFO] Clustering with kmeans...
[INFO] Silhouette Score: 0.1138

Silhouette Score: 0.1138


In [5]:
# Assign cluster names
cluster_names = cluster_analyzer.assign_cluster_names(X.toarray(), vectorizer)
print("\nCluster Names:")
for cid, name in cluster_names.items():
    print(f"  Cluster {cid}: {name}")


Cluster Names:
  Cluster 0: Topic: getaway, exceed, expect
  Cluster 1: Cleanliness & Room Quality
  Cluster 2: Topic: beauti, perfect, servic
  Cluster 3: Service & Staff Quality
  Cluster 4: Room & Comfort


In [6]:
# Cluster statistics
df_clustered = cluster_analyzer.add_clusters_to_dataframe(df_cleaned)
cluster_stats = cluster_analyzer.get_cluster_statistics(df_clustered)
print("\nCluster Statistics:")
cluster_stats


Cluster Statistics:


,cluster_id,cluster_name,n_reviews,percentage,avg_rating,rating_std,rating_distribution,avg_review_length,dominant_sentiment
0,0,"Topic: getaway, exceed, expect",89,4.45,4.561798,0.498978,"{5: 50, 4: 39}",84.348315,positive
1,1,Cleanliness & Room Quality,349,17.45,3.530086,1.112797,"{3: 147, 5: 92, 4: 64, 2: 29, 1: 17}",74.212034,positive
2,2,"Topic: beauti, perfect, servic",127,6.35,4.614173,0.488718,"{5: 78, 4: 49}",84.511811,positive
3,3,Service & Staff Quality,1236,61.80,3.316343,1.267887,"{3: 387, 5: 290, 4: 250, 2: 179, 1: 130}",77.824434,positive
4,4,Room & Comfort,199,9.95,4.170854,0.772609,"{5: 79, 4: 75, 3: 45}",81.743719,positive


In [7]:
# Visualize clusters
plot_gen = PlotGenerator()
fig = plot_gen.plot_cluster_visualization(coords_2d, labels, cluster_names)
import matplotlib.pyplot as plt
plt.show()
# Plot settings
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)


## 4. Save Results

In [8]:
# Save clustering results
df_clustered.to_csv('../data/processed/clustered_data.csv', index=False)
rules.to_csv('../data/processed/association_rules.csv', index=False)
print("Saved clustering and rules results")

Saved clustering and rules results
